# 技能3 · Day 4 上机：因果发现与 ML 因果推断

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实数据集）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 PC 算法在**真实数据**上自动发现因果图结构，区分因果发现与因果推断
2. 用因果森林（CausalForestDML）估计**异质处理效应（HTE/CATE）**，找出"对哪类用户效果最大"
3. 解释 PC/FCI 的假设差异（因果充分性）、DML 的去偏原理、因果森林的分裂标准

## 说明
本笔记本分两部分：
- **Part 1（TODO 1-3）**：因果发现 -- 用 PC 算法从 sklearn 糖尿病真实数据自动学因果图
- **Part 2（TODO 4-6）**：ML 因果推断 -- 用因果森林在 NSW 真实数据上估计异质处理效应

本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

In [ ]:
# !pip install causal-learn econml causaldata scikit-learn -q

## 1. 因果发现数据背景

**数据集**：sklearn 糖尿病数据集（真实医学数据，442 样本 x 10 特征）

| 特征 | 含义 | 营销映射 |
|------|------|---------|
| `age` | 年龄（标准化） | 用户年龄段 |
| `sex` | 性别 | 用户性别 |
| `bmi` | 体质指数 | 用户健康画像 |
| `bp` | 平均血压 | 用户健康指标 |
| `s1`-`s6` | 6项血液指标 | 用户行为特征 |

**因果发现问题**：这 10 个变量之间，谁影响谁？-- 不给算法任何先验知识，让它从数据中**自动发现因果结构**。

**营销映射**：在营销中，你有页面浏览、搜索、加购、优惠券使用、下单等行为数据，但不确定谁导致谁。因果发现 = 让算法**自动**从行为日志中学出因果图，替代人工猜测 DAG。

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from causaldata import nsw
from causallearn.search.ConstraintBased.PC import pc
from econml.dml import CausalForestDML
import warnings
warnings.filterwarnings('ignore')

## 2. PC 算法原理回顾

**PC 算法**（Peter-Clark）是最经典的因果发现算法：

1. **骨架学习**：从完全连接图开始，通过条件独立性检验逐步删除边
   - 若 X _||_ Y | S（存在条件集 S 使 X、Y 独立），则删除 X-Y 边
   - 从 |S|=0 开始逐步增加，直到 |S| 达到最大邻域大小
2. **方向定向**：用 v-结构（collider）检测确定箭头方向
   - X->Z<-Y 且 X、Y 不相邻 -> Z 是 collider，定向为 X->Z<-Y
   - 应用方向传播规则：避免产生新 v-结构和有向环

**PC 三大假设**：因果马尔可夫 + **因果充分性（无隐混杂）** + 忠实性

**FCI 扩展**：放宽因果充分性假设，允许隐混杂因素存在，输出 PAG（部分祖先图），用更丰富的边类型（o->, <->, o-o）表示不确定性

> 详细理论见 notes.md 理论部分与独立教材 4.1 节。

## Part 1: 因果发现（TODO 1-3）

In [ ]:
# TODO 1：加载真实糖尿病数据（因果发现用）
# 提示：load_diabetes() 返回对象有 .data 和 .feature_names
# 要求：加载为 DataFrame df_cd，打印形状和前5行

# ===== 你的代码 =====
df_cd = None  # TODO: 用 load_diabetes() 加载，转为 DataFrame
# ====================

print(f"数据形状: {df_cd.shape}")
print(f"特征: {list(df_cd.columns)}")
df_cd.head()

In [ ]:
# TODO 2：运行 PC 算法，自动发现因果图
# 提示：cg = pc(data_array, alpha=0.05)  # data_array 是 numpy 数组（df_cd.values）
# 要求：运行 PC，打印完成信息

# ===== 你的代码 =====
cg = None  # TODO: 调用 pc() 函数
# ====================

print("PC 算法完成")

In [ ]:
# TODO 3：提取并解读发现的因果结构
# 提示：cg.G.graph 是邻接矩阵（numpy 数组）
#       约定：graph[i,j]=1 且 graph[j,i]=-1 表示 i->j；graph[i,j]=-1 且 graph[j,i]=-1 表示无向边
# 要求：遍历所有变量对，打印有边的对及其方向；统计总边数

# ===== 你的代码 =====

# ====================

## 3. ML 因果推断数据背景

**数据集**：Lalonde/NSW 真实数据（同 Day 1），NSW 职业培训实验

| NSW 变量 | 营销映射 | 角色 |
|---------|---------|------|
| `treat` | 是否收到优惠券 | 处理 T（二元） |
| `re78` | 转化率/GMV | 结果 Y |
| `age`,`education`,`re74`,`re75`,... | 用户画像/历史消费 | 特征 X（异质性来源）|

**因果问题**：培训（优惠券）的平均效应（ATE）是多少？更重要的是--**对不同特征的用户，效应是否不同（CATE）？** 哪类用户收益最大？

**营销映射**：传统 A/B 测试告诉你"优惠券平均有效5%"；因果森林告诉你"对高活跃用户有效8%，对低活跃用户无效"--直接指导精准投放。

## 4. 因果森林与 DML 原理回顾

**因果森林**（Athey & Wager）：随机森林的因果推断版本，专攻**异质处理效应（CATE）**估计。

- **分裂标准**：不像普通随机森林最小化 Y 的预测误差，而是**最大化子节点间处理效应的差异**
- **CausalForestDML**：同时用 DML 去偏 + 因果森林分裂，兼顾 ATE 无偏和 CATE 异质性

**DML 去偏原理**（Double/Debiased ML）：
1. 用 ML 分别预测 Y|X 和 T|X，取残差 Y_tilde = Y - Y_hat, T_tilde = T - T_hat
2. 交叉拟合（cross-fitting）避免过拟合
3. 在残差上估计因果效应 = 无偏 ATE

> 详细理论见 notes.md 理论部分与独立教材 4.2-4.3 节。

## Part 2: ML 因果推断（TODO 4-6）

In [ ]:
# TODO 4：加载 NSW 真实数据（因果森林用）
# 提示：nsw.load_pandas().data
# 要求：加载为 df_ml，打印形状和前5行

# ===== 你的代码 =====
df_ml = None  # TODO
# ====================

print(f"数据形状: {df_ml.shape}")
df_ml.head()

In [ ]:
# TODO 5：用因果森林（CausalForestDML）估计异质处理效应
# 提示：
#   X = df_ml[['age','education','black','hispanic','married','nodegree','re74','re75']].values
#   T = df_ml['treat'].values; Y = df_ml['re78'].values
#   cf = CausalForestDML(model_y=RandomForestRegressor(n_estimators=50,max_depth=6),
#                        model_t=RandomForestClassifier(n_estimators=50,max_depth=6),
#                        n_estimators=200, min_samples_leaf=20, discrete_treatment=True)
#   cf.fit(Y, T, X=X); cate_pred = cf.effect(X=X)
# 要求：拟合模型，打印因果森林 ATE（=cate_pred 均值）和 CATE 范围

# ===== 你的代码 =====
cf = None        # TODO: 创建 CausalForestDML 并 fit
cate_pred = None  # TODO: 用 cf.effect() 估计每个样本的 CATE
# ====================

ate_cf = cate_pred.mean()
print(f"因果森林 ATE = {ate_cf:.2f}")
print(f"CATE 范围: [{cate_pred.min():.2f}, {cate_pred.max():.2f}]")

In [ ]:
# TODO 6：分析 CATE 异质性 -- 哪类用户对处理响应最大？
# 提示：
#   df_ml['cate'] = cate_pred
#   按 age 分组、按 re75（前期收入）分组，看 CATE 是否随特征变化
#   cf.feature_importances_ 给出哪些特征最影响处理效应的异质性
# 要求：(1) 按年龄分组打印平均 CATE  (2) 打印特征重要性

# ===== 你的代码 =====

# ====================

## 5. 反思与前沿

### 反思问题
1. PC 算法发现的因果图中有哪些边是**符合医学常识**的？有哪些是**反直觉**的？（反直觉的边可能是伪相关或隐混杂）
2. PC 假设因果充分性（无隐混杂），如果糖尿病数据中有未观测的混杂因素（如"基因"），PC 的发现还可靠吗？-> 这正是 FCI 存在的理由
3. 因果森林发现哪类用户对培训（优惠券）的响应最大？`feature_importances_` 排第一的特征是什么？这对精准投放有什么启示？
4. 对比 Day 1 的后门调整 ATE 和今天的因果森林 ATE，它们接近吗？如果不接近，为什么？

### 2026 前沿：LLM 辅助因果发现

传统因果发现（PC/FCI/NOTEARS）纯靠数据驱动，不利用领域知识。2026 年的新趋势是用 **LLM 辅助因果发现**：

- **思路**：用 LLM 从领域知识/文献/文本中提取因果图候选（"什么可能导致什么"），再与数据驱动的因果发现结果**融合**
- LLM 提供先验因果方向（如"BMI 影响血压"而非反过来），数据驱动方法验证/修正这些先验
- 参考 Kiciman et al. (2023) "Causal Reasoning and Large Language Models"（arXiv 2305.00050）--微软研究院发现 LLM 在因果图构建任务上达到或超过人类专家水平

**注意**：LLM 辅助因果发现仍处于研究阶段。LLM 可能产生"幻觉因果边"，必须与数据驱动方法交叉验证。把它定位为"提供候选因果图的助手"，最终因果结构需数据验证。

> 深入阅读见 reading.md 的 LLM 辅助因果发现条目。